In [0]:
from databricks.feature_engineering import FeatureEngineeringClient
import pyspark.sql.functions as F

fe = FeatureEngineeringClient()

# 1. El Caos (Sin Feature Store)
# Equipo A calcula el scoring de un cliente así:
# df_equipo_a = spark.sql("SELECT user_id, count(transaction_id) as tx_count FROM transactions GROUP BY user_id")
# Equipo B lo hace diferente en otro notebook... ¡Riesgo de inconsistencia!

# 2. La Solución: Crear y registrar features centralizadas
# Imaginemos que ya tenemos un DataFrame pulido y curado
customer_features_df = spark.createDataFrame([
    (1, 35, 120.50, "activo"),
    (2, 42, 85.00, "inactivo"),
    (3, 28, 900.20, "activo")
], ["user_id", "age", "avg_ticket", "status"])

# Guardamos la tabla en Unity Catalog como Feature Table
table_name = "mlops_dbx_talk_dev.ezapata.customer_features_demo"
fe.create_table(
    name=table_name,
    primary_keys=["user_id"],
    df=customer_features_df,
    description="Características demográficas y de comportamiento de clientes."
)
print(f"¡Feature Table {table_name} creada con éxito en Unity Catalog!")

# 3. Consumo por parte de un Data Scientist para un modelo
# El Data Scientist solo tiene los IDs y lo que quiere predecir (Target)
training_labels = spark.createDataFrame([
    (1, 1), # user_id, compró (target)
    (2, 0)
], ["user_id", "target"])

# Mágicamente traemos las features sin hacer JOINs manuales complejos
from databricks.feature_engineering import FeatureLookup

lookups = [
    FeatureLookup(
        table_name=table_name,
        feature_names=["age", "avg_ticket"], # Solo traemos lo que necesitamos
        lookup_key="user_id"
    )
]

training_set = fe.create_training_set(
    df=training_labels,
    feature_lookups=lookups,
    label="target"
)

display(training_set.load_df())
print("El modelo ahora tiene el contexto completo sin código espagueti.")